# 02. Satellite Feature Engineering
This notebook processes features from satellite instruments: CAMS AOD, MAIAC AOD, FIRMS Fire Detections, and Sentinel-5P HCHO.

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
import glob

ROOT = Path("../../")
RAW_DIR = ROOT / "data/raw"
PROC_DIR = ROOT / "data/processed"
PROC_DIR.mkdir(parents=True, exist_ok=True)

STATIONS = {
    2161292: (21.0152, 105.7999),
    2161306: (21.0500, 105.7400),
    4946811: (21.0491, 105.8831),
    4946812: (21.0031, 105.7947),
    4946813: (21.0052, 105.8418),
    6123215: (20.9933, 105.9441),
}


## 1. Process CAMS AOD
Extract daily mean, max, and morning values from cached hourly CAMS data.

In [2]:
all_records = []
for loc_id, (lat, lon) in STATIONS.items():
    cache_file = RAW_DIR / "cams" / f"aod_{loc_id}.csv"
    if not cache_file.exists():
        continue
    df_hr = pd.read_csv(cache_file, parse_dates=["time"])
    df_hr["time"] = pd.to_datetime(df_hr["time"])
    df_hr["date"] = df_hr["time"].dt.normalize()
    df_hr["is_morning"] = df_hr["time"].dt.hour.isin([6, 7, 8, 9])

    daily = (df_hr.groupby("date")["aod"]
             .agg(aod_550_mean="mean", aod_550_max="max")
             .reset_index())
    morn = (df_hr[df_hr["is_morning"]]
             .groupby("date")["aod"].mean()
             .rename("aod_550_morning").reset_index())
    daily = daily.merge(morn, on="date", how="left")
    daily.insert(0, "location_id", loc_id)
    all_records.append(daily)

if all_records:
    aod_df = (pd.concat(all_records, ignore_index=True)
              .sort_values(["location_id", "date"])
              .reset_index(drop=True))
    aod_df["month"] = aod_df["date"].dt.month
    aod_df.to_csv(PROC_DIR / "07_cams_aod_daily.csv", index=False)
    aod_df.to_csv(PROC_DIR / "cams_aod_daily.csv", index=False)
    print(f"Saved CAMS AOD daily. Shape: {aod_df.shape}")


Saved CAMS AOD daily. Shape: (5196, 6)


## 2. Process MAIAC AOD
Calculates Ångström Exponent (between 550nm and 470nm bands) and formats columns.

In [3]:
all_dfs = []
for loc_id in STATIONS:
    cache_file = RAW_DIR / "maiac" / f"maiac_{loc_id}.csv"
    if not cache_file.exists():
        continue
    df_st = pd.read_csv(cache_file)
    df_st["date"] = df_st["date"].astype(str).str.strip()
    all_dfs.append(df_st)

if all_dfs:
    aod_df = (pd.concat(all_dfs, ignore_index=True)
              .sort_values(["location_id", "date"])
              .reset_index(drop=True))
    
    if 'aod047_mean' in aod_df.columns:
        LOG_RATIO = np.log(550 / 470)
        mask = (aod_df['aod_mean'] > 0.01) & (aod_df['aod047_mean'] > 0.01)
        aod_df['angstrom_exp'] = np.nan
        aod_df.loc[mask, 'angstrom_exp'] = (
            np.log(aod_df.loc[mask, 'aod_mean'] / aod_df.loc[mask, 'aod047_mean']) / LOG_RATIO
        ).clip(-0.1, 2.5)

    rename_map = {
        "aod_mean": "maiac_aod_mean",
        "aod_median": "maiac_aod_median",
        "aod_max": "maiac_aod_max",
        "aod_count": "maiac_aod_count",
    }
    aod_merge = aod_df.rename(columns=rename_map)
    
    keep_cols = ["location_id", "date", "maiac_aod_mean", "maiac_aod_median", "maiac_aod_count"]
    aod_merge = aod_merge[keep_cols]
    
    aod_merge.to_csv(PROC_DIR / "08_maiac_aod_daily.csv", index=False)
    aod_merge.to_csv(PROC_DIR / "maiac_aod_daily.csv", index=False)
    print(f"Saved MAIAC AOD daily. Shape: {aod_merge.shape}")


Saved MAIAC AOD daily. Shape: (4842, 5)


## 3. Process FIRMS Fire Detections
Calculates distances to station coordinates and aggregates number of fires and FRP values within 100km radius, filtering out low confidence observations.

In [4]:
def haversine_km(lat1, lon1, lat2_arr, lon2_arr):
    R = 6371.0
    dlat = np.radians(lat2_arr - lat1)
    dlon = np.radians(lon2_arr - lon1)
    a = np.sin(dlat/2)**2 + np.cos(np.radians(lat1)) * np.cos(np.radians(lat2_arr)) * np.sin(dlon/2)**2
    return R * 2 * np.arcsin(np.sqrt(a))

csv_files = sorted(glob.glob(str(RAW_DIR / "firms/**/*.csv"), recursive=True))
frames = []
for f in csv_files:
    df = pd.read_csv(f)
    if "acq_date" in df.columns:
        df["acq_date"] = pd.to_datetime(df["acq_date"])
    else:
        date_col = [c for c in df.columns if "date" in c.lower()][0]
        df["acq_date"] = pd.to_datetime(df[date_col])
    frames.append(df)

if frames:
    fire_raw = pd.concat(frames, ignore_index=True)
    BBOX = {"west": 98, "south": 13, "east": 117, "north": 28}
    fire_raw = fire_raw[
        (fire_raw["latitude"] >= BBOX["south"]) & (fire_raw["latitude"] <= BBOX["north"]) &
        (fire_raw["longitude"] >= BBOX["west"]) & (fire_raw["longitude"] <= BBOX["east"])
    ].reset_index(drop=True)

    if "confidence" in fire_raw.columns:
        fire_raw = fire_raw[fire_raw["confidence"] != "l"].reset_index(drop=True)

    fire_lats = fire_raw["latitude"].values
    fire_lons = fire_raw["longitude"].values
    fire_dates = fire_raw["acq_date"].values.astype("datetime64[D]")
    fire_frp = fire_raw["frp"].values if "frp" in fire_raw.columns else np.ones(len(fire_raw))

    max_date = pd.to_datetime(fire_dates.max())
    date_range = pd.date_range("2024-01-01", max_date, freq="D")

    records = []
    for loc_id, (st_lat, st_lon) in STATIONS.items():
        dists = haversine_km(st_lat, st_lon, fire_lats, fire_lons)
        for d in date_range:
            d64 = np.datetime64(d, "D")
            mask_d = fire_dates == d64
            mask = mask_d & (dists <= 100)
            cnt = int(mask.sum())
            if cnt >= 1:
                records.append({
                    "location_id": loc_id,
                    "date": d,
                    "fire_count_100km": cnt,
                    "frp_sum_100km": float(fire_frp[mask].sum()),
                    "frp_mean_100km": float(fire_frp[mask].mean()),
                    "frp_max_100km": float(fire_frp[mask].max())
                })

    fire_df = pd.DataFrame(records)
    fire_df["date"] = pd.to_datetime(fire_df["date"])
    
    fire_df.to_csv(PROC_DIR / "09_firms_daily.csv", index=False)
    fire_df.to_csv(PROC_DIR / "firms_daily.csv", index=False)
    print(f"Saved FIRMS Fire daily. Shape: {fire_df.shape}")


Saved FIRMS Fire daily. Shape: (2963, 6)


## 4. Process Sentinel-5P HCHO
Concatenates per-station cached HCHO daily CSVs.

In [5]:
hcho_files = sorted(glob.glob(str(RAW_DIR / "hcho_s5p/hcho_s5p_*.csv")))
frames = []
for f in hcho_files:
    df_st = pd.read_csv(f)
    frames.append(df_st)

if frames:
    hcho_df = pd.concat(frames, ignore_index=True)
    hcho_df["date"] = pd.to_datetime(hcho_df["date"])
    hcho_df = hcho_df.sort_values(["location_id", "date"]).reset_index(drop=True)
    hcho_df["month"] = hcho_df["date"].dt.month
    
    keep_cols = ["hcho_valid_pixels", "hcho_mean", "hcho_std", "hcho_min", "hcho_max", "hcho_median", "location_id", "date", "month"]
    hcho_df = hcho_df[keep_cols]
    
    hcho_df.to_csv(PROC_DIR / "10_hcho_s5p_daily.csv", index=False)
    hcho_df.to_csv(PROC_DIR / "hcho_s5p_daily.csv", index=False)
    print(f"Saved HCHO daily. Shape: {hcho_df.shape}")


Saved HCHO daily. Shape: (4842, 9)
